In [0]:
dbutils.widgets.text("database_name",'',"database_name")
databaseName = dbutils.widgets.get("database_name")

dbutils.widgets.text("retention_period_in_hrs",'170',"retention_period_in_hrs")
retentionPeriod = int(dbutils.widgets.get("retention_period_in_hrs"))

dbutils.widgets.text("no_parallel_runs",'10',"no_parallel_runs")
parallelVacuumCount = int(dbutils.widgets.get("no_parallel_runs"))

print(f"databaseName -- {databaseName}")
print(f"retentionPeriod -- {retentionPeriod}")
print(f"parallelVacuumCount -- {parallelVacuumCount}")

In [0]:
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, ArrayType
#####
schema = StructType([StructField('name',StringType(),False),StructField('catalog',StringType(),True),StructField('namespace',ArrayType(StringType()),True),StructField('description',StringType(),True),StructField('tableType',StringType(),True),StructField('isTemporary',BooleanType(),True)])
tablesDetailedList = spark.createDataFrame(spark.catalog.listTables(databaseName),schema=schema).filter("tableType != 'VIEW' and isTemporary == False").collect()

In [0]:
#Function to perform vacuum on a table 
def executeVacuumOnDelta(databaseName,tableName,retentionPeriod):
  output = (f"{databaseName}.{tableName}",'FAILURE')
  try:
    tableDetails = spark.sql(f"describe detail {databaseName}.{tableName}").collect()
    if tableDetails[0]['format'] == 'delta' and tableDetails[0]['location'].startswith('abfs'):
      print(f"VACUUM  - {databaseName}.{tableName}")
      spark.sql(f"VACUUM {databaseName}.{tableName} RETAIN {retentionPeriod} HOURS ")
      print(f"VACUUM COMPLETED SUCCESSFULLY FOR - {databaseName}.{tableName}")
      output = (f"{databaseName}.{tableName}",'SUCCESS')
  except Exception as e:
    print(f"VACUUM FAILED FOR - {databaseName}.{tableName}")
  return output

#Function for multi threading of vacuum process
def executeParallel(tableList , parallelCount):
  with ThreadPoolExecutor(max_workers = parallelCount) as executor:
      return [executor.submit(executeVacuumOnDelta,databaseName,tableName,retentionPeriod) for tableName in tableList]  

In [0]:
tableNameList = [table['name'] for table in tablesDetailedList ]
results = executeParallel(tableNameList,int(parallelVacuumCount))

In [0]:
resultsDf = spark.createDataFrame([i.result() for i in results],['tableName','vacuumStatus'])
resultsDf.display()